# 03 - Claim a bounded work batch

This dispatcher notebook grants leases without exceeding the configured active-worker limit. Claims are conditional Delta updates. A retry with the same `DISPATCHER_ID` returns the same unexpired claims instead of allocating more work.

In [ ]:
DISPATCHER_ID = ""
MAX_CONCURRENT_WORKERS = 4
CLAIM_LIMIT = 4
LEASE_MINUTES = 30
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import random
import re
import time
import uuid

from delta.tables import DeltaTable
import notebookutils
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
dispatcher_id = DISPATCHER_ID.strip()
if not dispatcher_id:
    raise ValueError("DISPATCHER_ID must be a stable pipeline run ID")
max_workers = int(MAX_CONCURRENT_WORKERS)
claim_limit = int(CLAIM_LIMIT)
lease_minutes = int(LEASE_MINUTES)
if max_workers < 1 or claim_limit < 1 or lease_minutes < 5:
    raise ValueError("Worker and claim limits must be positive; LEASE_MINUTES must be at least 5")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
work_table = table("video_work")
attempts_table = table("video_attempts")
dispatcher_leases_table = table("dispatcher_leases")
now = datetime.now(timezone.utc)
lease_expires = now + timedelta(minutes=lease_minutes)
active_states = ["LEASED", "STAGING", "RUNNING", "WRITING"]


def retry_delta(operation, attempts: int = 6) -> None:
    for number in range(attempts):
        try:
            operation()
            return
        except Exception as error:
            message = f"{type(error).__name__}: {error}".lower()
            retryable = any(token in message for token in ("concurrent", "conflict", "changedexception"))
            if not retryable or number == attempts - 1:
                raise
            time.sleep((2 ** number) * 0.25 + random.random() * 0.5)


lock_expires = now + timedelta(minutes=5)
lock_source = spark_session.createDataFrame(
    [("global", dispatcher_id, now, lock_expires)],
    "lock_name string, owner_id string, acquired_at timestamp, expires_at timestamp",
)
retry_delta(
    lambda: (
        DeltaTable.forName(spark_session, dispatcher_leases_table)
        .alias("t")
        .merge(lock_source.alias("s"), "t.lock_name = s.lock_name")
        .whenMatchedUpdateAll(condition="t.expires_at <= current_timestamp() OR t.owner_id = s.owner_id")
        .whenNotMatchedInsertAll()
        .execute()
    )
)
lock_row = spark_session.table(dispatcher_leases_table).where(F.col("lock_name") == "global").limit(1).collect()
if len(lock_row) != 1 or lock_row[0].owner_id != dispatcher_id:
    raise RuntimeError("Another dispatcher currently owns the global claim mutex")

owned = (
    spark_session.table(work_table)
    .where(
        (F.col("lease_dispatcher_id") == dispatcher_id)
        & (F.col("status") == "LEASED")
        & (F.col("lease_expires_at") > F.lit(now))
    )
    .select("work_id", F.col("lease_owner_attempt_id").alias("attempt_id"))
)
owned_rows = owned.collect()

if owned_rows:
    claimed = owned
    active_before = None
else:
    active_before = (
        spark_session.table(work_table)
        .where(F.col("status").isin(active_states) & (F.col("lease_expires_at") > F.lit(now)))
        .count()
    )
    available = max(0, max_workers - active_before)
    take = min(available, claim_limit)
    if take == 0:
        claimed = spark_session.createDataFrame([], "work_id string, attempt_id string")
    else:
        candidate_rows = (
            spark_session.table(work_table)
            .where(
                F.col("status").isin("QUEUED", "RETRY_WAIT")
                & (F.col("attempt_count") < F.col("max_attempts"))
                & (F.col("not_before_at").isNull() | (F.col("not_before_at") <= F.lit(now)))
                & (F.col("lease_expires_at").isNull() | (F.col("lease_expires_at") <= F.lit(now)))
            )
            .orderBy(F.col("priority").desc(), F.col("queued_at").asc(), F.col("work_id"))
            .limit(take)
            .select("work_id", "capture_date")
            .collect()
        )
        source = spark_session.createDataFrame(
            [
                (row.work_id, row.capture_date, uuid.uuid4().hex, dispatcher_id, lease_expires)
                for row in candidate_rows
            ],
            "work_id string, capture_date date, attempt_id string, dispatcher_id string, lease_expires timestamp",
        )
        retry_delta(
            lambda: (
                DeltaTable.forName(spark_session, work_table)
                .alias("t")
                .merge(source.alias("s"), "t.work_id = s.work_id AND t.capture_date = s.capture_date")
                .whenMatchedUpdate(
                    condition=(
                        "t.status IN ('QUEUED', 'RETRY_WAIT') AND "
                        "t.attempt_count < t.max_attempts AND "
                        "(t.not_before_at IS NULL OR t.not_before_at <= current_timestamp()) AND "
                        "(t.lease_expires_at IS NULL OR t.lease_expires_at <= current_timestamp())"
                    ),
                    set={
                        "status": "'LEASED'",
                        "attempt_count": "t.attempt_count + 1",
                        "lease_owner_attempt_id": "s.attempt_id",
                        "lease_dispatcher_id": "s.dispatcher_id",
                        "lease_acquired_at": "current_timestamp()",
                        "lease_expires_at": "s.lease_expires",
                        "last_heartbeat_at": "current_timestamp()",
                        "not_before_at": "NULL",
                    },
                )
                .execute()
            )
        )
        claimed = (
            spark_session.table(work_table)
            .where(
                (F.col("lease_dispatcher_id") == dispatcher_id)
                & (F.col("status") == "LEASED")
                & (F.col("lease_expires_at") > F.lit(now))
            )
            .select("work_id", F.col("lease_owner_attempt_id").alias("attempt_id"))
        )

attempt_rows = (
    claimed.alias("c")
    .join(spark_session.table(work_table).alias("w"), "work_id")
    .select(
        F.col("c.attempt_id"),
        F.col("work_id"),
        F.lit(dispatcher_id).alias("dispatcher_id"),
        F.lit(None).cast("string").alias("pipeline_run_id"),
        F.lit(None).cast("string").alias("activity_run_id"),
        F.lit(None).cast("string").alias("fabric_job_instance_id"),
        F.lit(None).cast("string").alias("sdk_version"),
        F.lit(None).cast("string").alias("bundle_manifest_sha256"),
        F.col("w.config_sha256"),
        F.lit("LEASED").alias("status"),
        F.col("w.lease_acquired_at").alias("claimed_at"),
        F.lit(None).cast("timestamp").alias("staging_started_at"),
        F.lit(None).cast("timestamp").alias("inference_started_at"),
        F.lit(None).cast("timestamp").alias("writing_started_at"),
        F.lit(None).cast("timestamp").alias("completed_at"),
        F.col("w.last_heartbeat_at"),
        F.lit(None).cast("string").alias("input_sha256"),
        F.lit(None).cast("long").alias("source_size_bytes"),
        F.lit(None).cast("double").alias("source_duration_seconds"),
        F.lit(None).cast("double").alias("source_fps"),
        F.lit(None).cast("long").alias("total_source_frames"),
        F.lit(None).cast("long").alias("processed_frames"),
        F.lit(None).cast("double").alias("effective_sample_fps"),
        F.lit(None).cast("double").alias("processing_seconds"),
        F.lit(None).cast("long").alias("distinct_people"),
        F.lit(None).cast("long").alias("line_in_count"),
        F.lit(None).cast("long").alias("line_out_count"),
        F.lit(None).cast("boolean").alias("retryable"),
        F.lit(None).cast("string").alias("error_category"),
        F.lit(None).cast("string").alias("error_type"),
        F.lit(None).cast("string").alias("error_message"),
        F.col("w.capture_date"),
    )
)
retry_delta(
    lambda: (
        DeltaTable.forName(spark_session, attempts_table)
        .alias("t")
        .merge(attempt_rows.alias("s"), "t.attempt_id = s.attempt_id AND t.capture_date = s.capture_date")
        .whenNotMatchedInsertAll()
        .execute()
    )
)

items = [row.asDict(recursive=True) for row in claimed.orderBy("work_id").collect()]
outcome = {
    "dispatcher_id": dispatcher_id,
    "active_before": active_before,
    "claimed_count": len(items),
    "items": items,
}
retry_delta(
    lambda: DeltaTable.forName(spark_session, dispatcher_leases_table).delete(
        (F.col("lock_name") == "global") & (F.col("owner_id") == dispatcher_id)
    )
)
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))